# `tmp_imbalance_analysis.py` — Rare-Class F1 Quick Report

## Purpose
A lightweight **scratch script** used to compare the effect of different loss functions and augmentation strategies on **rare/imbalanced classes** across experiments.

## What it does
- Loads `results/experiments/all_results.json` (the combined experiment results file).
- Prints a formatted comparison table of Macro-F1, Accuracy, and per-class F1 scores for the rarest classes:
  - `price-neg` (very few negative-sentiment price reviews)
  - `price-neu` (very few neutral price reviews)
  - `packing-neu` (few neutral packing reviews)
  - `smell-neu` (few neutral smell reviews)

## When to use
Run this after adding new experiments to `all_results.json` to immediately see whether a new loss configuration improved rare-class performance.

In [ ]:
import json

# Load the full experiment results file
# This is populated by experiment_runner.py after each training run
d = json.load(open('results/experiments/all_results.json'))

## Define the Experiments to Compare
These are the experiment IDs from the ablation study (A3 = loss function ablations, A7 = hybrid weight tuning, A4 = augmentation).

In [ ]:
# List of (experiment_id, human-readable label) pairs to include in the report
experiments = [
    ('A3_ce_loss',           'CE Loss (no imbalance handling)'),
    ('A3_focal_only',        'Focal Loss only'),
    ('A3_cb_only',           'CB Loss only'),
    ('A3_dice_only',         'Dice Loss only'),
    ('A3_hybrid_loss',       'Hybrid: Focal+CB+Dice (original)'),
    ('A7_hybrid_cb_05',      'Hybrid: Focal+CB 0.5 (no Dice) [NEW]'),
    ('A7_hybrid_cb_10',      'Hybrid: Focal+CB 1.0 (no Dice)'),
    ('A4_no_augmentation',   'Full model WITHOUT augmentation'),
    ('A4_with_augmentation', 'Full model WITH augmentation'),
]

## Print the Comparison Table

In [ ]:
# Print header row
print(f"{'Experiment':<42} | {'Macro-F1':^10} | {'Accuracy':^10} | {'price-neg':^10} | {'price-neu':^10} | {'pack-neu':^10} | {'smell-neu':^10}")
print('-' * 115)

for exp_id, label in experiments:
    r          = d.get(exp_id, {})      # Look up this experiment's data (empty dict if missing)
    overall    = r.get('overall', {})   # Top-level aggregate metrics
    f1         = overall.get('macro_f1', None)
    acc        = overall.get('accuracy', None)
    per_aspect = r.get('per_aspect', {})  # Broken down by each aspect

    def rare_f1(aspect, cls_idx):
        """Look up the F1 for a specific class within a specific aspect.
        cls_idx: 0=negative, 1=neutral, 2=positive
        """
        asp_data = per_aspect.get(aspect, {})
        pcf1     = asp_data.get('per_class_f1', [])  # [neg_f1, neu_f1, pos_f1]
        if len(pcf1) > cls_idx:
            return f'{pcf1[cls_idx]:.4f}'
        return '  N/A '

    # Extract rare-class F1 scores
    price_neg = rare_f1('price',   0)   # price-negative class F1
    price_neu = rare_f1('price',   1)   # price-neutral class F1
    pack_neu  = rare_f1('packing', 1)   # packing-neutral class F1
    smell_neu = rare_f1('smell',   1)   # smell-neutral class F1

    # Format floats cleanly, fall back to 'N/A' if the experiment hasn't run yet
    f1_str  = f'{f1:.4f}'  if isinstance(f1,  float) else '  N/A  '
    acc_str = f'{acc:.4f}' if isinstance(acc, float) else '  N/A  '

    print(f"{label:<42} | {f1_str:^10} | {acc_str:^10} | {price_neg:^10} | {price_neu:^10} | {pack_neu:^10} | {smell_neu:^10}")